# Irrigation Water Requirement Prediction Dataset
In this notebook, we train a neural network to predict the amount of irrigation required under varying environmental conditions.

For more information, look up the dataset description at:  
https://www.kaggle.com/datasets/miadul/irrigation-water-requirement-prediction-dataset

-------
## Initialization

In [1]:
# imports
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm

In [2]:
# load data
df = pd.read_csv("irrigation.csv")

In [3]:
# hyperparameters
RANDOM_STATE = 42
BATCH_SIZE = 32
EPOCHS = 10
LR = 0.01

In [4]:
# label mapping
LABEL_MAP = {'Low': 0, 'Medium': 1, 'High': 2}

-------
## EDA

In [5]:
print('Shape:', df.shape)
print('\nColumn dtypes:')
print(df.dtypes)
print('\nDescribe (numeric):')
print(df.describe())
print('\nDescribe (all):')
print(df.describe(include='all'))

Shape: (10000, 20)

Column dtypes:
Soil_Type                   object
Soil_pH                    float64
Soil_Moisture              float64
Organic_Carbon             float64
Electrical_Conductivity    float64
Temperature_C              float64
Humidity                   float64
Rainfall_mm                float64
Sunlight_Hours             float64
Wind_Speed_kmh             float64
Crop_Type                   object
Crop_Growth_Stage           object
Season                      object
Irrigation_Type             object
Water_Source                object
Field_Area_hectare         float64
Mulching_Used               object
Previous_Irrigation_mm     float64
Region                      object
Irrigation_Need             object
dtype: object

Describe (numeric):
            Soil_pH  Soil_Moisture  Organic_Carbon  Electrical_Conductivity  \
count  10000.000000   10000.000000    10000.000000             10000.000000   
mean       6.487857      36.969207        0.944731                 1.791

In [6]:
# Categorical counts
obj_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
bool_cols = df.select_dtypes(include=['bool']).columns.tolist()
if obj_cols:
    print('\nCategorical column value counts:')
    for c in obj_cols:
        print(f"\n-- {c} --")
        print(df[c].value_counts(dropna=False))
if bool_cols:
    print('\nBoolean columns:')
    for c in bool_cols:
        print(f"{c}:\n", df[c].value_counts(dropna=False))


Categorical column value counts:

-- Soil_Type --
Soil_Type
Sandy    2536
Silt     2501
Loamy    2486
Clay     2477
Name: count, dtype: int64

-- Crop_Type --
Crop_Type
Rice         1711
Maize        1694
Sugarcane    1678
Potato       1663
Wheat        1659
Cotton       1595
Name: count, dtype: int64

-- Crop_Growth_Stage --
Crop_Growth_Stage
Harvest       2581
Vegetative    2521
Flowering     2465
Sowing        2433
Name: count, dtype: int64

-- Season --
Season
Rabi      3383
Kharif    3362
Zaid      3255
Name: count, dtype: int64

-- Irrigation_Type --
Irrigation_Type
Sprinkler    2527
Rainfed      2511
Drip         2495
Canal        2467
Name: count, dtype: int64

-- Water_Source --
Water_Source
River          2528
Reservoir      2513
Rainwater      2497
Groundwater    2462
Name: count, dtype: int64

-- Mulching_Used --
Mulching_Used
No     5013
Yes    4987
Name: count, dtype: int64

-- Region --
Region
South      2102
West       2017
East       1994
Central    1987
North      19

In [7]:
# Target distribution
if 'Irrigation_Need' in df.columns:
    print('\nTarget distribution (Irrigation_Need):')
    print(df['Irrigation_Need'].value_counts(normalize=False))


Target distribution (Irrigation_Need):
Irrigation_Need
Low       5864
Medium    3800
High       336
Name: count, dtype: int64


-------
## Data wrangling helper functions

In [ ]:
# prepare training df with only numeric features
def prepare_numeric(df):
    df = df.copy()
    df = df.dropna()  # drop rows with missing values
    y = df['Irrigation_Need'].map(LABEL_MAP).astype(int)
    X = df.drop(columns=['Irrigation_Need'])  # drop target column
    X_num = X.select_dtypes(include=[np.number])
    return X_num, y

In [ ]:
# prepare training df with ALL features (including categoricals)
def prepare_with_categoricals(df):
    df = df.copy()
    df = df.dropna()  # drop rows with missing values
    y = df['Irrigation_Need'].map(LABEL_MAP).astype(int)
    X = df.drop(columns=['Irrigation_Need'])  # drop target column

    # Convert bools to ints
    bool_cols = X.select_dtypes(include=['bool']).columns.tolist()
    for c in bool_cols:
        X[c] = X[c].astype(int)

    # One-hot encode object/category columns
    X = pd.get_dummies(X, drop_first=False)
    return X, y

-------
## Training the neural network

In [10]:
## model definition
class SmallNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 10),
            nn.ELU(),
            nn.Linear(10, 15),
            nn.ELU(),
            nn.Linear(15, 3)  # 3 classes
        )

    def forward(self, x):
        return self.net(x)

In [11]:
def train_and_validate(X, y, device, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR):
    # Split: 60% train, 20% val, 20% test
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=0.25, random_state=RANDOM_STATE, stratify=y_trainval)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    # Convert to tensors
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train.values, dtype=torch.long)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.values, dtype=torch.long)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    y_test_t = torch.tensor(y_test.values, dtype=torch.long)

    train_ds = TensorDataset(X_train_t, y_train_t)
    val_ds = TensorDataset(X_val_t, y_val_t)
    test_ds = TensorDataset(X_test_t, y_test_t)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=(device.type=='cuda'))
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=(device.type=='cuda'))

    model = SmallNN(input_dim=X_train.shape[1]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    print(f"\nTraining model (input_dim={X_train.shape[1]}) on device: {device}")

    for epoch in range(1, epochs+1):
        model.train()
        running_loss = 0.0
        for xb, yb in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}"):
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
        avg_train_loss = running_loss / len(train_loader.dataset)

        # Validation
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
                preds = logits.argmax(dim=1)
                correct += (preds == yb).sum().item()
                total += xb.size(0)
        avg_val_loss = val_loss / len(val_loader.dataset)
        val_acc = correct / total if total>0 else 0.0
        print(f"Epoch {epoch}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}, Val Acc={val_acc:.4f}")

    # Return model and test dataset for final evaluation
    return model, scaler, test_ds

In [12]:
# Let's determine the device to use (GPU if available, else CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [13]:
# 1) Numeric-only features
print('\n\n--- Training with numeric-only features ---')
X_num, y_num = prepare_numeric(df)
print('Numeric feature columns:', X_num.columns.tolist())
if X_num.shape[1] == 0:
    print('No numeric features found. Skipping numeric-only training.')
else:
    model_num, scaler_num, test_ds_num = train_and_validate(X_num, y_num, device)



--- Training with numeric-only features ---
Numeric feature columns: ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']

Training model (input_dim=11) on device: cuda


Epoch 1/10: 100%|██████████| 188/188 [00:00<00:00, 326.93it/s]


Epoch 1: Train Loss=0.6711, Val Loss=0.6531, Val Acc=0.6780


Epoch 2/10: 100%|██████████| 188/188 [00:00<00:00, 454.11it/s]


Epoch 2: Train Loss=0.6364, Val Loss=0.6297, Val Acc=0.6980


...

Epoch 8: Train Loss=0.5940, Val Loss=0.6052, Val Acc=0.7140


Epoch 9/10: 100%|██████████| 188/188 [00:00<00:00, 464.67it/s]


Epoch 9: Train Loss=0.5912, Val Loss=0.6010, Val Acc=0.7115


Epoch 10/10: 100%|██████████| 188/188 [00:00<00:00, 454.18it/s]


Epoch 10: Train Loss=0.5843, Val Loss=0.5895, Val Acc=0.7225


In [14]:
# 2) Include categorical features (one-hot)
print('\n\n--- Training including categorical features (one-hot encoded) ---')
X_all, y_all = prepare_with_categoricals(df)
print('Feature columns after encoding:', X_all.shape[1])
model_all, scaler_all, test_ds_all = train_and_validate(X_all, y_all, device)



--- Training including categorical features (one-hot encoded) ---
Feature columns after encoding: 43

Training model (input_dim=43) on device: cuda


Epoch 1/10: 100%|██████████| 188/188 [00:00<00:00, 402.77it/s]


Epoch 1: Train Loss=0.4597, Val Loss=0.4026, Val Acc=0.8215


Epoch 2/10: 100%|██████████| 188/188 [00:00<00:00, 427.50it/s]


Epoch 2: Train Loss=0.3949, Val Loss=0.4063, Val Acc=0.8195


...

Epoch 8: Train Loss=0.2365, Val Loss=0.2586, Val Acc=0.8855


Epoch 9/10: 100%|██████████| 188/188 [00:00<00:00, 398.69it/s]


Epoch 9: Train Loss=0.2199, Val Loss=0.2489, Val Acc=0.8925


Epoch 10/10: 100%|██████████| 188/188 [00:00<00:00, 407.59it/s]


Epoch 10: Train Loss=0.2074, Val Loss=0.2525, Val Acc=0.8900


Additionally including the categorical features into the model significantly improved the model performance as per the loss and accuracy on the validation. As such, we'll keep this model for evaluation.

-------
## Evaluation

In [15]:
def evaluate_model(model, scaler, test_ds, device):
    model.eval()
    X_test_t, y_test_t = test_ds.tensors
    X_test_t = X_test_t.to(device)
    y_test_t = y_test_t.to(device)
    with torch.no_grad():
        logits = model(X_test_t)
        preds = logits.argmax(dim=1).cpu().numpy()
        truths = y_test_t.cpu().numpy()

    y_true = truths
    y_pred = preds
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("\nClassification Report:")
    classes = ['Low', 'Medium', 'High']
    print(classification_report(y_true, y_pred, target_names=classes))
    print('\nConfusion Matrix (rows=true, cols=pred):')
    print(confusion_matrix(y_true, y_pred))

In [16]:
# Final evaluation on test set with the final model (model_all)
evaluate_model(model_all, scaler_all, test_ds_all, device)

Accuracy: 0.902

Classification Report:
              precision    recall  f1-score   support

         Low       0.92      0.94      0.93      1173
      Medium       0.88      0.86      0.87       760
        High       0.96      0.64      0.77        67

    accuracy                           0.90      2000
   macro avg       0.92      0.82      0.86      2000
weighted avg       0.90      0.90      0.90      2000


Confusion Matrix (rows=true, cols=pred):
[[1105   68    0]
 [ 102  656    2]
 [   0   24   43]]
